# TCC — Raio-X 2D: pré-rótulo NOVO (fase 1) e retreino com flip que troca o rótulo (fase 2)


**Antes de rodar**
- **Acelerador:** GPU T4 x1. **Internet:** ligada (o Detectron2 é instalado do GitHub).
- **Inputs:** `dentex-challenge-2023`, o dataset privado com as panorâmicas pré-processadas (nome em `DATASET_PANORAMICAS`, na célula de configuração; as 4 pastas `*_limpo`, **incluindo `processados4_limpo`**) e `xraymodeltcc` (pesos atuais).
- **1ª vez:** rode com `TESTE_RAPIDO = True` (célula de configuração), ~30 min. Valida as duas fases com 12 imagens e 200 iterações de treino.
- **Rodada completa:** mude para `TESTE_RAPIDO = False` → *Save Version* → *Save & Run All*. Leva ~7 h: fase 1 ~1 h, treino ~4 h, avaliações ~30 min e, se o modelo novo for melhor, mais ~1 h para regerar os pré-rótulos.

**Fase 1 — pré-rótulo NOVO, sem treinar** (modelo atual)
recorte das barras pretas + ampliação para 1333 px → sem CLAHE extra → lado pela linha média → NMS agnóstico + 1 detecção por número + resgate.
Saída: `pseudo_labels_novo.zip` (8.352 JSONs, mesmo formato de antes, coordenadas na imagem 512).

**Fase 2 — retreino**
1. Avalia o modelo atual no DENTEX val (127 imagens): linha de base.
2. Testa o `FlipSwapMapper` antes de treinar (o flip tem de trocar Q1↔Q2 e Q4↔Q3).
3. Treina o mesmo Mask R-CNN (mesma configuração do treino original) com o flip que troca o rótulo.
4. Avalia o modelo novo do mesmo jeito e compara.
5. Se o modelo novo for melhor, regera os pré-rótulos com ele: `pseudo_labels_novo_v2.zip`.

**Saídas** (`/kaggle/working`): `pseudo_labels_novo.zip`, `fase2_resultados.json`, `output_maskrcnn_flipswap/model_final.pth` e, se houver, `pseudo_labels_novo_v2.zip`.

In [ ]:
%%bash
set -e  # para imediatamente se qualquer comando falhar

if python -c "import detectron2" 2>/dev/null; then
    echo "Detectron2 já instalado nesta sessão."
    exit 0
fi

echo "=== [1/3] Detectando arquitetura da GPU ==="
export TORCH_CUDA_ARCH_LIST=$(python -c "import torch; cap = torch.cuda.get_device_capability(0); print(f'{cap[0]}.{cap[1]}')")
echo "-> Arquitetura para compilação: $TORCH_CUDA_ARCH_LIST"

echo "=== [2/3] Compilando o Detectron2 (~10 min) ==="
pip uninstall -y detectron2 iopath 2>/dev/null || true
pip install "iopath==0.1.10" --no-deps -q
FORCE_CUDA="1" pip install --no-cache-dir -q 'git+https://github.com/facebookresearch/detectron2.git'

echo "=== [3/3] Verificando ambiente ==="
python - <<'EOF'
import torch, detectron2
print(f"GPU        : {torch.cuda.get_device_name(0)}")
print(f"PyTorch    : {torch.__version__} | CUDA {torch.version.cuda}")
print(f"Detectron2 : {detectron2.__version__}")
EOF

In [ ]:
import PIL.Image

# O Detectron2 ainda usa nomes antigos do Pillow (removidos no Pillow 10+).
for antigo, novo in [("LINEAR", "BILINEAR"), ("CUBIC", "BICUBIC"), ("ANTIALIAS", "LANCZOS")]:
    if not hasattr(PIL.Image, antigo):
        setattr(PIL.Image, antigo, getattr(PIL.Image, novo))
print("Patch do Pillow aplicado.")

## Configuração

In [ ]:
import os, json, random, time, shutil, copy, contextlib, io
from pathlib import Path

import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# ── O que rodar ──────────────────────────────────────────────────────────────
TESTE_RAPIDO        = True   # True: 12 imagens, 200 iterações, 10 imagens de val (valida o notebook, ~30 min)
                             # False: rodada completa (8.352 imagens, 20.000 iterações, 127 imagens de val)
RODAR_FASE1         = True   # pré-rótulo NOVO com o modelo atual
RODAR_FASE2         = True   # linha de base → teste do mapper → treino → avaliação → comparação
REGERAR_SE_MELHORAR = True   # fase 2: se o modelo novo for melhor no DENTEX val, regera os pré-rótulos com ele

# ── Caminhos ─────────────────────────────────────────────────────────────────
DENTEX_ROOT = "/kaggle/input/datasets/truthisneverlinear/dentex-challenge-2023"
MEUS_DADOS  = "/kaggle/input/datasets/thiagokeishimasaki"
DATASET_PANORAMICAS = "panoramicas-tcc"   # dataset privado com as panorâmicas pré-processadas (512×512)

PATHS = {
    # DENTEX (só para preparar o split, se ele não estiver no xraymodeltcc)
    "xrays_enumeration": f"{DENTEX_ROOT}/training_data/training_data/quadrant_enumeration/xrays",
    "ann_enumeration":   f"{DENTEX_ROOT}/training_data/training_data/quadrant_enumeration/train_quadrant_enumeration.json",

    # Panorâmicas do TCC — as 4 pastas revisadas (3.781 + 1.974 + 1.959 + 638 = 8.352)
    "ds_images": [
        f"{MEUS_DADOS}/{DATASET_PANORAMICAS}/processados_limpo",
        f"{MEUS_DADOS}/{DATASET_PANORAMICAS}/processados2_limpo",
        f"{MEUS_DADOS}/{DATASET_PANORAMICAS}/processados3_limpo",
        f"{MEUS_DADOS}/{DATASET_PANORAMICAS}/processados4_limpo",
    ],

    # Modelo atual e o split DENTEX usado no treino dele (507 treino / 127 val)
    "pesos_atual":            f"{MEUS_DADOS}/xraymodeltcc/KaggleResults/output_maskrcnn/model_final.pth",
    "dentex_preparado_input": f"{MEUS_DADOS}/xraymodeltcc/KaggleResults/dentex_maskrcnn",

    # Saídas (graváveis)
    "dentex_preparado":   "/kaggle/working/dentex_maskrcnn",
    "pseudo_novo":        "/kaggle/working/pseudo_labels_novo",
    "pseudo_novo_v2":     "/kaggle/working/pseudo_labels_novo_v2",
    "treino_out":         "/kaggle/working/output_maskrcnn_flipswap" + ("_teste" if TESTE_RAPIDO else ""),
    "fase2_resultados":   "/kaggle/working/fase2_resultados.json",
}

# ── Constantes ───────────────────────────────────────────────────────────────
# 32 classes = 4 quadrantes × 8 posições (FDI). category_id COCO 1..32 = q*8 + pos + 1;
# id contíguo do Detectron2 0..31 = q*8 + pos  (q: 0=Q1, 1=Q2, 2=Q3, 3=Q4).
NUM_CLASSES       = 32
SCORE_THRESH_TEST = 0.05   # o modelo devolve tudo acima disso (precisa para o mAP e para o resgate)
SCORE_MIN         = 0.50   # limiar dos pré-rótulos (0,3–0,6 não muda nada depois das correções)
LARGURA_AMPLIADA  = 1333   # largura do conteúdo depois do recorte (mesma escala do treino)
CLAHE = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))  # só para preparar o DENTEX e para a condição "atual"

# ── Se um caminho não existir, procura pelo nome em /kaggle/input ────────────
def _achar(caminho, profundidade=6):
    """Devolve o caminho configurado se existir; senão, o primeiro item com o mesmo nome em /kaggle/input."""
    if os.path.exists(caminho):
        return caminho
    nome = Path(caminho).name
    base = Path("/kaggle/input")
    for raiz, dirs, files in os.walk(base):
        if len(Path(raiz).relative_to(base).parts) >= profundidade:
            dirs[:] = []
            continue
        if nome in dirs or nome in files:
            achado = os.path.join(raiz, nome)
            print(f"  [caminho ajustado] {nome}: {achado}")
            return achado
    return caminho

PATHS["ds_images"] = [_achar(d) for d in PATHS["ds_images"]]
for k in ["pesos_atual", "dentex_preparado_input", "xrays_enumeration", "ann_enumeration"]:
    PATHS[k] = _achar(PATHS[k])

# ── Conferência dos inputs ───────────────────────────────────────────────────
print("Inputs:")
for k in ["pesos_atual", "dentex_preparado_input", "xrays_enumeration", "ann_enumeration"]:
    ok = os.path.exists(PATHS[k])
    obs = ""
    if not ok and k in ("xrays_enumeration", "ann_enumeration") and os.path.exists(PATHS["dentex_preparado_input"]):
        obs = "  (não precisa: o split salvo do xraymodeltcc foi encontrado)"
    print(f"  {k:<24} {'OK ' if ok else 'NÃO ENCONTRADO'}  {PATHS[k]}{obs}")
for d in PATHS["ds_images"]:
    n = sum(1 for p in Path(d).rglob("*") if p.suffix.lower() in {".png", ".jpg", ".jpeg"}) if os.path.exists(d) else 0
    print(f"  {Path(d).name:<24} {n:>5} imagens  {'' if n else '<-- NÃO ENCONTRADO: anexe ao dataset ' + DATASET_PANORAMICAS}")
if not all(os.path.exists(d) for d in PATHS["ds_images"]):
    print("\nO que está montado em /kaggle/input (até 3 níveis):")
    for raiz, dirs, _ in os.walk("/kaggle/input"):
        nivel = len(Path(raiz).relative_to("/kaggle/input").parts)
        if nivel >= 3:
            dirs[:] = []
        print("   " + "  " * nivel + (Path(raiz).name or "/kaggle/input") + "/")
print(f"\nGPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NENHUMA (ligue a T4)'}")
print(f"Modo: {'TESTE RÁPIDO' if TESTE_RAPIDO else 'RODADA COMPLETA'}")

## Pipeline NOVO de inferência

Pós-processamento (score, NMS, numeração ordenada e correção de lado) que levou o F1 de 34% → 91% no DENTEX val (50 imagens).

1. **Recorte + ampliação:** tira as barras pretas do letterbox 512 e amplia o conteúdo para 1333 px de largura. O problema do 512 era o tamanho aparente dos dentes, não a falta de informação.
2. **Sem CLAHE extra:** as panorâmicas do TCC já saem do `01_dados/preprocessar_panoramica.py` com CLAHE.
3. **Lado pela linha média:** mantém a arcada e a posição (1–8) do modelo e redefine o quadrante pelo centróide. Esquerda da imagem → Q1/Q4; direita → Q2/Q3 (98% dos dentes do DENTEX seguem essa regra).
4. **NMS agnóstico à classe (IoU 0,5) + 1 detecção por número.**
5. **Resgate:** uma detecção descartada por número repetido, que não se sobrepõe a nenhum dente mantido, recebe o número livre entre os vizinhos com maior probabilidade.

In [ ]:
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor
from torchvision.ops import nms
from pycocotools import mask as mask_util


def imread(p):
    """Leitura segura (funciona também com acentos no caminho)."""
    return cv2.imdecode(np.fromfile(str(p), np.uint8), cv2.IMREAD_COLOR)


def install_prob_hook(predictor):
    """Guarda as probabilidades das 32 classes de cada detecção (usadas no resgate)."""
    bp = predictor.model.roi_heads.box_predictor
    orig = bp.inference

    def inference(predictions, proposals):
        probs = bp.predict_probs(predictions, proposals)          # lista [R, K+1]
        instances, kept = orig(predictions, proposals)
        for inst, pr, k in zip(instances, probs, kept):
            inst.probs = pr[k][:, :-1]                              # sem o fundo
        return instances, kept
    bp.inference = inference


def build_predictor(weights):
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
    cfg.MODEL.ROI_HEADS.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = SCORE_THRESH_TEST
    cfg.MODEL.WEIGHTS = str(weights)
    cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    pred = DefaultPredictor(cfg)
    install_prob_hook(pred)
    return pred


def predict(pred, img):
    """Saída crua do modelo. classes em 1..32."""
    inst = pred(img)["instances"].to("cpu")
    return dict(boxes=inst.pred_boxes.tensor.numpy(), scores=inst.scores.numpy(),
                classes=inst.pred_classes.numpy() + 1, masks=inst.pred_masks.numpy(),
                probs=inst.probs.numpy())


def letterbox512(img):
    """Igual ao letterbox de 01_dados/preprocessar_panoramica.py. Retorna a imagem e (escala, pad_x, pad_y)."""
    h, w = img.shape[:2]
    s = min(512 / w, 512 / h)
    nw, nh = int(w * s), int(h * s)
    r = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
    out = np.zeros((512, 512, 3), np.uint8)
    px, py = (512 - nw) // 2, (512 - nh) // 2
    out[py:py + nh, px:px + nw] = r
    return out, (s, px, py)


def crop_upscale(img, target_w=LARGURA_AMPLIADA):
    """Remove as barras pretas do letterbox e amplia o conteúdo até target_w de largura.
    Retorna a imagem ampliada e (escala, x0, y0, largura_recorte, altura_recorte)."""
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    rows = np.where(g.max(axis=1) > 10)[0]
    cols = np.where(g.max(axis=0) > 10)[0]
    if len(rows) == 0 or len(cols) == 0:                      # imagem toda preta: não recorta
        rows, cols = np.arange(img.shape[0]), np.arange(img.shape[1])
    y0, y1, x0, x1 = rows[0], rows[-1] + 1, cols[0], cols[-1] + 1
    c = img[y0:y1, x0:x1]
    s = target_w / c.shape[1]
    up = cv2.resize(c, None, fx=s, fy=s, interpolation=cv2.INTER_CUBIC)
    return up, (s, int(x0), int(y0), int(x1 - x0), int(y1 - y0))


def to_input_space(p, s, x0, y0, H, W, score_min):
    """Filtra por score e leva máscaras e caixas da imagem ampliada de volta à imagem de entrada (H x W)."""
    sel = p["scores"] >= score_min
    p = {k: v[sel] for k, v in p.items()}
    masks = []
    for m in p["masks"]:
        m = m.astype(np.uint8)
        h, w = m.shape
        small = cv2.resize(m, (int(round(w / s)), int(round(h / s))), interpolation=cv2.INTER_NEAREST)
        full = np.zeros((H, W), np.uint8)
        hh, ww = min(small.shape[0], H - y0), min(small.shape[1], W - x0)
        full[y0:y0 + hh, x0:x0 + ww] = small[:hh, :ww]
        masks.append(full)
    p["masks"] = np.array(masks) if masks else np.zeros((0, H, W), np.uint8)
    b = p["boxes"] / s + np.array([x0, y0, x0, y0], np.float32)
    p["boxes"] = np.clip(b, 0, [W, H, W, H]).astype(np.float32)
    return p


def postprocess(p, iou=0.5):
    """NMS agnóstico à classe + no máximo 1 detecção por número FDI (a de maior score)."""
    if len(p["scores"]) == 0:
        return p
    keep = nms(torch.tensor(p["boxes"]), torch.tensor(p["scores"]), iou).numpy()
    best = {}
    for i in keep:
        c = p["classes"][i]
        if c not in best or p["scores"][i] > p["scores"][best[c]]:
            best[c] = i
    idx = np.array(sorted(best.values(), key=lambda i: -p["scores"][i]))
    return {k: v[idx] for k, v in p.items()}


def fix_side(classes, cx, mid_x):
    """Mantém arcada e posição; quadrante pelo lado da linha média.
    Esquerda da imagem -> Q1 (sup) / Q4 (inf); direita -> Q2 (sup) / Q3 (inf)."""
    upper = np.isin((classes - 1) // 8, [0, 1])
    pos = (classes - 1) % 8
    left = cx < mid_x
    q = np.where(upper, np.where(left, 0, 1), np.where(left, 3, 2))
    return q * 8 + pos + 1


def number_with_rescue(p, mid_x, usar_linha_media=True, overlap_max=0.3):
    """Lado pela linha média + NMS agnóstico + 1 por número e depois RESGATE: uma detecção descartada
    por número repetido que não se sobrepõe a nenhum dente mantido recebe o número livre (entre os
    vizinhos, na ordem a partir da linha média) de maior probabilidade. Não muda dentes já mantidos."""
    if len(p["scores"]) == 0:
        return p
    cx = np.array([np.nonzero(m)[1].mean() if m.any() else (b[0] + b[2]) / 2
                   for m, b in zip(p["masks"], p["boxes"])])
    cls0 = fix_side(p["classes"], cx, mid_x) if usar_linha_media else p["classes"].copy()
    q = dict(p, classes=cls0, idx=np.arange(len(cx)))
    kept = postprocess(q)
    kset = set(kept["idx"].tolist())
    cls = {int(i): int(c) for i, c in zip(kept["idx"], kept["classes"])}
    km = {i: p["masks"][i].astype(bool) for i in kset}
    pr = p["probs"].reshape(-1, 4, 8)
    cand = sorted((i for i in range(len(cx)) if i not in kset), key=lambda i: -p["scores"][i])
    for i in cand:
        m = p["masks"][i].astype(bool)
        if any((m & km[j]).sum() / max((m | km[j]).sum(), 1) > overlap_max for j in km):
            continue                                          # é o mesmo dente de alguém mantido
        quad = (int(cls0[i]) - 1) // 8
        same = [j for j in cls if (cls[j] - 1) // 8 == quad]
        d = abs(cx[i] - mid_x)
        inner = [(cls[j] - 1) % 8 for j in same if abs(cx[j] - mid_x) < d]
        outer = [(cls[j] - 1) % 8 for j in same if abs(cx[j] - mid_x) > d]
        lo = max(inner) + 1 if inner else 0
        hi = min(outer) - 1 if outer else 7
        used = {(cls[j] - 1) % 8 for j in same}
        free = [ps for ps in range(lo, hi + 1) if ps not in used]
        if not free:
            continue
        arch = [0, 1] if quad in (0, 1) else [2, 3]
        pp = pr[i, arch].sum(0)
        ps = max(free, key=lambda k: pp[k])
        cls[i] = quad * 8 + ps + 1
        km[i] = m
    order = sorted(cls)
    out = {k: v[order] for k, v in p.items()}
    out["classes"] = np.array([cls[i] for i in order])
    return out


def novo_bruto(pred, img):
    """Recorte + ampliação + inferência, com tudo devolvido no espaço da imagem de entrada.
    Retorna (predições com score >= SCORE_MIN, x da linha média)."""
    H, W = img.shape[:2]
    up, (s, x0, y0, cw, ch) = crop_upscale(img)
    p = to_input_space(predict(pred, up), s, x0, y0, H, W, SCORE_MIN)
    return p, x0 + cw / 2


def predict_novo(pred, img, usar_linha_media=True):
    """Pipeline NOVO completo."""
    p, mid_x = novo_bruto(pred, img)
    return number_with_rescue(p, mid_x, usar_linha_media)


def to_annotations(p):
    """Predições -> anotações no formato dos JSONs de pré-rótulo (category_id 1..32, bbox xywh)."""
    anns = []
    for b, s, c, m in zip(p["boxes"], p["scores"], p["classes"], p["masks"]):
        contours, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        seg = [ct.flatten().tolist() for ct in contours if len(ct) >= 3]
        if not seg:
            continue
        x1, y1, x2, y2 = [float(v) for v in b]
        anns.append({"category_id": int(c), "bbox": [x1, y1, x2 - x1, y2 - y1],
                     "segmentation": seg, "score": float(s)})
    return anns

print("Funções do pipeline NOVO carregadas.")

## Fase 1 — pré-rótulo NOVO com o modelo atual

É **resumível**: pula imagens cujo JSON já existe. Se a sessão cair, rode a célula de novo.
O nome de cada JSON é `<pasta>__<caminho relativo>.json`, igual ao da rodada anterior. A saída fica numa pasta nova (`pseudo_labels_novo`), sem misturar com os pré-rótulos antigos.

In [ ]:
def _coletar_imagens(img_dirs, por_pasta=None):
    """Lista (raiz, caminho) de todas as imagens (busca recursiva). por_pasta limita quantas por pasta."""
    exts = {".png", ".jpg", ".jpeg"}
    items = []
    for d in img_dirs:
        root = Path(d)
        if not root.exists():
            print(f"[AVISO] pasta não encontrada: {root}")
            continue
        files = sorted(p for p in root.rglob("*") if p.suffix.lower() in exts)
        if por_pasta is not None:
            files = files[:por_pasta]
        print(f"  {root.name}: {len(files)} imagens")
        items += [(root, p) for p in files]
    return items


def prelabel_novo(img_dirs, out_dir, weights, por_pasta=None, usar_linha_media=True):
    """Roda o pipeline NOVO em cada imagem e salva um JSON por imagem (coordenadas da imagem de entrada, 512)."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    print("Coletando imagens...")
    items = _coletar_imagens(img_dirs, por_pasta)
    if not items:
        raise FileNotFoundError(f"Nenhuma imagem encontrada em: {img_dirs}")
    print(f"Total a processar: {len(items)}")

    pred = build_predictor(weights)
    salvas = existentes = sem_deteccao = erro_leitura = 0
    n_dentes = []
    for root, img_path in tqdm(items, desc="Pré-rótulo NOVO"):
        key = root.name + "__" + img_path.relative_to(root).with_suffix("").as_posix().replace("/", "_")
        out_file = out_dir / (key + ".json")
        if out_file.exists():
            existentes += 1
            continue
        img = imread(img_path)
        if img is None:
            erro_leitura += 1
            continue
        anns = to_annotations(predict_novo(pred, img, usar_linha_media))
        if not anns:
            sem_deteccao += 1
            continue
        result = {
            "file_name":   img_path.name,
            "source_path": str(img_path),
            "height":      img.shape[0],
            "width":       img.shape[1],
            "annotations": anns,
            "pipeline":    "novo: recorte+ampliação 1333, sem CLAHE extra, "
                           + ("lado pela linha média, " if usar_linha_media else "")
                           + "NMS agnóstico + 1 por número + resgate",
            "pesos":       str(weights),
        }
        with open(out_file, "w") as f:
            json.dump(result, f)
        salvas += 1
        n_dentes.append(len(anns))

    print(f"\nConcluído. {salvas} novas | {existentes} já existiam | {sem_deteccao} sem detecção | "
          f"{erro_leitura} falha de leitura  (total {len(items)})")
    if n_dentes:
        print(f"Dentes por imagem (novas): mediana {np.median(n_dentes):.0f}, "
              f"mín {min(n_dentes)}, máx {max(n_dentes)}")
    print(f"JSONs em: {out_dir}")
    del pred
    torch.cuda.empty_cache()


def _fdi(cid):
    return ((cid - 1) // 8 + 1) * 10 + (cid - 1) % 8 + 1


def visualizar_jsons(pseudo_dir, num_samples=6):
    """Desenha máscaras + número FDI sobre a imagem (via source_path)."""
    jsons = sorted(Path(pseudo_dir).glob("*.json"))
    if not jsons:
        print(f"Nenhum JSON em: {pseudo_dir}")
        return
    random.Random(0).shuffle(jsons)
    rng = np.random.RandomState(42)
    colors = rng.randint(60, 230, size=(33, 3))
    for jp in jsons[:num_samples]:
        data = json.load(open(jp))
        img = imread(data["source_path"])
        if img is None:
            print(f"[aviso] não abriu: {data['source_path']}")
            continue
        overlay = img.copy()
        for ann in data["annotations"]:
            color = [int(c) for c in colors[ann["category_id"]]]
            for poly in ann["segmentation"]:
                if len(poly) >= 6:
                    cv2.fillPoly(overlay, [np.array(poly, np.int32).reshape(-1, 1, 2)], color)
        blend = cv2.addWeighted(overlay, 0.45, img, 0.55, 0)
        for ann in data["annotations"]:
            x, y, w, h = [int(v) for v in ann["bbox"]]
            cv2.putText(blend, str(_fdi(ann["category_id"])), (x + w // 2 - 7, y + h // 2 + 4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.38, (255, 255, 255), 1, cv2.LINE_AA)
        mid = img.shape[1] // 2
        cv2.line(blend, (mid, 0), (mid, img.shape[0]), (0, 255, 255), 1)
        plt.figure(figsize=(12, 12))
        plt.imshow(cv2.cvtColor(blend, cv2.COLOR_BGR2RGB))
        plt.title(f"{jp.name} — {len(data['annotations'])} dentes  (linha amarela = meio da imagem)")
        plt.axis("off")
        plt.show()


def zipar(pasta):
    n = len(list(Path(pasta).glob("*.json")))
    zip_path = shutil.make_archive(str(pasta), "zip", pasta)
    print(f"OK: {n} JSONs em {zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB)")
    return zip_path

In [ ]:
if RODAR_FASE1:
    prelabel_novo(PATHS["ds_images"], PATHS["pseudo_novo"], PATHS["pesos_atual"],
                  por_pasta=3 if TESTE_RAPIDO else None)
    visualizar_jsons(PATHS["pseudo_novo"], num_samples=4 if TESTE_RAPIDO else 6)
    zipar(PATHS["pseudo_novo"])
else:
    print("Fase 1 desligada (RODAR_FASE1 = False).")

## Fase 2 — retreino com flip que troca o rótulo

### 2.1 Dataset DENTEX (mesmo split do modelo atual)

Usa o `dentex_maskrcnn` salvo no `xraymodeltcc`, se estiver lá: é exatamente o split (507/127) e o pré-processamento (CLAHE 1x + lado maior ≤ 1333) do treino original.
Se não estiver, refaz com o mesmo código e a mesma semente.

In [ ]:
from detectron2.data.datasets import register_coco_instances
from detectron2.data import DatasetCatalog, MetadataCatalog

PREP_MAX_SIZE = 1333


def preparar_dentex():
    src = Path(PATHS["dentex_preparado_input"])
    if (src / "annotations/instances_val2017.json").exists() and (src / "images").exists():
        print(f"Usando o split salvo do treino original: {src}")
        return src
    dst = Path(PATHS["dentex_preparado"])
    if (dst / "annotations/instances_val2017.json").exists():
        print(f"Usando o split já preparado nesta sessão: {dst}")
        return dst

    print("Split salvo não encontrado no xraymodeltcc; preparando de novo (semente 42, 80/20)...")
    img_dir, ann_dir = dst / "images", dst / "annotations"
    img_dir.mkdir(parents=True, exist_ok=True)
    ann_dir.mkdir(parents=True, exist_ok=True)
    raw = json.load(open(PATHS["ann_enumeration"]))
    categories = [{"id": i + 1, "name": f"tooth_{i + 1}", "supercategory": "tooth"} for i in range(NUM_CLASSES)]

    ann_by_image = {}
    for a in raw["annotations"]:
        ann_by_image.setdefault(a["image_id"], []).append({
            "id": a["id"], "image_id": a["image_id"],
            "category_id": a["category_id_1"] * 8 + a["category_id_2"] + 1,
            "bbox": list(a["bbox"]), "segmentation": [list(s) for s in a["segmentation"]],
            "area": a["area"], "iscrowd": a.get("iscrowd", 0),
        })

    all_ids = [img["id"] for img in raw["images"]]
    random.seed(42)
    random.shuffle(all_ids)
    cut = int(0.8 * len(all_ids))
    splits = {"train": set(all_ids[:cut]), "val": set(all_ids[cut:])}

    for name, ids in splits.items():
        imgs, anns = [], []
        for info in tqdm(raw["images"], desc=f"CLAHE+resize {name}"):
            if info["id"] not in ids:
                continue
            g = cv2.imread(os.path.join(PATHS["xrays_enumeration"], info["file_name"]), cv2.IMREAD_GRAYSCALE)
            if g is None:
                continue
            h, w = g.shape
            s = min(PREP_MAX_SIZE / max(h, w), 1.0)
            if s < 1.0:
                g = cv2.resize(g, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)
            cv2.imwrite(str(img_dir / info["file_name"]), cv2.cvtColor(CLAHE.apply(g), cv2.COLOR_GRAY2BGR))
            imgs.append(dict(info, width=g.shape[1], height=g.shape[0]))
            for a in ann_by_image.get(info["id"], []):
                a = dict(a)
                if s < 1.0:
                    x, y, bw, bh = a["bbox"]
                    a["bbox"] = [x * s, y * s, bw * s, bh * s]
                    a["area"] = a["area"] * s * s
                    a["segmentation"] = [[v * s for v in poly] for poly in a["segmentation"]]
                anns.append(a)
        json.dump({"images": imgs, "annotations": anns, "categories": categories},
                  open(ann_dir / f"instances_{name}2017.json", "w"))
        print(f"  {name}: {len(imgs)} imagens, {len(anns)} anotações")
    return dst


if RODAR_FASE2:
    DENTEX_DIR = preparar_dentex()
    for split in ("train", "val"):
        name = f"dentex_{split}"
        if name not in DatasetCatalog.list():
            register_coco_instances(name, {}, str(DENTEX_DIR / f"annotations/instances_{split}2017.json"),
                                    str(DENTEX_DIR / "images"))
    VAL_JSON = str(DENTEX_DIR / "annotations/instances_val2017.json")
    print(f"train: {len(DatasetCatalog.get('dentex_train'))} imagens | val: {len(DatasetCatalog.get('dentex_val'))} imagens"
          "  (esperado 507 / 127)")

### 2.2 Avaliação no DENTEX val

Em todas as condições, um dente conta como acerto quando tem o mesmo número FDI e IoU de máscara ≥ 0,5. P/R/F1 usam score ≥ 0,5.

| Condição | O que mede |
|---|---|
| **A. cru, resolução do treino** | o modelo em si (+ mAP de máscara, score ≥ 0,05) |
| **B. NOVO a partir do 512** | o pré-rótulo de verdade, nas condições das panorâmicas do TCC (**métrica principal**) |
| **C. NOVO sem linha média** | se a correção de lado ainda faz diferença depois do retreino |
| **D. atual (512 + CLAHE 2x, cru)** | o pipeline antigo, para conferir os 34% |

O mAP desta célula usa CLAHE 1x; os 0,425 da avaliação antiga do Kaggle usavam CLAHE 2x. Os dois modelos são avaliados aqui do mesmo jeito, então a comparação entre eles é justa.

In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


def _rle(m):
    r = mask_util.encode(np.asfortranarray(m.astype(np.uint8)))
    r["counts"] = r["counts"].decode()
    return r


def gt_rles(coco, img_id, H, W, tf=None):
    """GT como [(category_id, rle)]; tf=(s, px, py) leva os polígonos para o espaço letterbox 512."""
    out = []
    for a in coco.loadAnns(coco.getAnnIds(imgIds=img_id)):
        segs = a["segmentation"]
        if tf is not None:
            s, px, py = tf
            segs = [[v * s + (px if i % 2 == 0 else py) for i, v in enumerate(poly)] for poly in segs]
        out.append((a["category_id"], mask_util.merge(mask_util.frPyObjects(segs, H, W))))
    return out


def match(p, gts, thr=0.5, score_min=SCORE_MIN):
    """TP/FP/FN por dente. Também conta 'lado errado' (mesma posição, quadrante contralateral)."""
    sel = p["scores"] >= score_min
    pc = p["classes"][sel]
    n_gt = len(gts)
    tp = side = other = 0
    if len(pc) and n_gt:
        ious = np.asarray(mask_util.iou([_rle(m) for m in p["masks"][sel]], [g[1] for g in gts], [0] * n_gt))
        used = set()
        for i, c in enumerate(pc):
            bj = int(np.argmax(ious[i]))
            best = ious[i, bj]
            gc = gts[bj][0]
            if best >= thr and gc == c and bj not in used:
                tp += 1
                used.add(bj)
            elif best >= thr:
                if (c - 1) % 8 == (gc - 1) % 8 and {(c - 1) // 8, (gc - 1) // 8} in ({0, 1}, {2, 3}):
                    side += 1
                else:
                    other += 1
    return dict(tp=tp, fp=len(pc) - tp, fn=n_gt - tp, side=side, other=other, ngt=n_gt)


def f1(counts):
    tp = sum(c["tp"] for c in counts); fp = sum(c["fp"] for c in counts); fn = sum(c["fn"] for c in counts)
    P = tp / max(tp + fp, 1); R = tp / max(tp + fn, 1)
    return P, R, 2 * P * R / max(P + R, 1e-9)


def resumo(counts):
    P, R, F = f1(counts)
    return dict(P=P, R=R, F1=F, lado_errado=sum(c["side"] for c in counts),
                outra_posicao=sum(c["other"] for c in counts), dentes=sum(c["ngt"] for c in counts))


def mask_map(results, img_ids):
    if not results:
        return 0.0, 0.0
    with contextlib.redirect_stdout(io.StringIO()):
        gt = COCO(VAL_JSON)
        dt = gt.loadRes(results)
        e = COCOeval(gt, dt, "segm")
        e.params.imgIds = img_ids
        e.evaluate(); e.accumulate(); e.summarize()
    return float(e.stats[0]), float(e.stats[1])


def clahe2x(img):
    return cv2.cvtColor(CLAHE.apply(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)), cv2.COLOR_GRAY2BGR)


def avaliar_modelo(weights, img_ids):
    """Avalia um modelo nas condições A–D. Devolve o resumo e as contagens por imagem (para o IC)."""
    coco = COCO(VAL_JSON)
    img_dir = DENTEX_DIR / "images"
    pred = build_predictor(weights)
    por_img = {k: [] for k in "ABCD"}
    results_A = []
    for iid in tqdm(img_ids, desc=f"Avaliando {Path(weights).parent.name}"):
        info = coco.loadImgs(iid)[0]
        img = imread(img_dir / info["file_name"])
        H, W = img.shape[:2]
        gts_full = gt_rles(coco, iid, H, W)
        lb, tf = letterbox512(img)
        gts_512 = gt_rles(coco, iid, 512, 512, tf)

        pa = predict(pred, img)                                       # A
        por_img["A"].append(match(pa, gts_full))
        for b, s, c, m in zip(pa["boxes"], pa["scores"], pa["classes"], pa["masks"]):
            results_A.append(dict(image_id=iid, category_id=int(c), score=float(s), segmentation=_rle(m)))

        p, mid_x = novo_bruto(pred, lb)                               # B e C (uma inferência só)
        por_img["B"].append(match(number_with_rescue(p, mid_x, True), gts_512))
        por_img["C"].append(match(number_with_rescue(p, mid_x, False), gts_512))

        por_img["D"].append(match(predict(pred, clahe2x(lb)), gts_512))  # D
    del pred
    torch.cuda.empty_cache()

    res = {
        "A_cru_resolucao_treino":   resumo(por_img["A"]),
        "B_novo_512":               resumo(por_img["B"]),
        "C_novo_512_sem_linha_media": resumo(por_img["C"]),
        "D_atual_512_clahe2x":      resumo(por_img["D"]),
    }
    res["A_cru_resolucao_treino"]["mAP_mascara"], res["A_cru_resolucao_treino"]["AP50_mascara"] = \
        mask_map(results_A, img_ids)
    return res, por_img


def imprimir(nome, res):
    print(f"\n{nome}")
    for cond, r in res.items():
        extra = f" | mAP máscara={r['mAP_mascara']:.3f} AP50={r['AP50_mascara']:.3f}" if "mAP_mascara" in r else ""
        print(f"  {cond:<28} P={r['P']:6.1%}  R={r['R']:6.1%}  F1={r['F1']:6.1%}  "
              f"| lado errado={r['lado_errado']:4d}  outra posição={r['outra_posicao']:4d}{extra}")


if RODAR_FASE2:
    VAL_IDS = sorted(COCO(VAL_JSON).getImgIds())
    if TESTE_RAPIDO:
        VAL_IDS = VAL_IDS[:10]
    print(f"Avaliação em {len(VAL_IDS)} imagens do DENTEX val.")

In [ ]:
if RODAR_FASE2:
    t0 = time.time()
    RES_ATUAL, POR_IMG_ATUAL = avaliar_modelo(PATHS["pesos_atual"], VAL_IDS)
    imprimir(f"MODELO ATUAL — linha de base ({len(VAL_IDS)} imagens, {(time.time() - t0) / 60:.1f} min)", RES_ATUAL)
    print("\nReferência do diagnóstico (50 imagens): B ≈ 90,9%, D ≈ 34,0%.")

### 2.3 `FlipSwapMapper` — conferência antes de treinar

O flip horizontal padrão do Detectron2 espelha a imagem, mas mantém o número do dente. Um 11 vai parar do lado onde fica o 21, e o modelo aprende que esquerda e direita são aleatórias.
O `FlipSwapMapper` troca o quadrante sempre que a imagem é espelhada: Q1↔Q2 e Q4↔Q3, ou seja, id contíguo `q*8+pos` → `(q^1)*8+pos`.

**Teste:** em imagens do treino, mede a fração de dentes de Q1/Q4 que aparecem na **esquerda** da imagem depois do mapper.
- `DatasetMapper` padrão: ~50% (metade das imagens é espelhada sem trocar o número).
- `FlipSwapMapper`: ~98% (como no GT, sem flip).

A célula para com erro se o `FlipSwapMapper` não passar.

In [ ]:
import detectron2.data.transforms as T
from detectron2.data import DatasetMapper, build_detection_train_loader
from detectron2.engine import DefaultTrainer
from detectron2.evaluation import COCOEvaluator


class FlipSwapMapper(DatasetMapper):
    """Flip horizontal que também espelha a numeração FDI (Q1<->Q2, Q4<->Q3)."""
    def _transform_annotations(self, dataset_dict, transforms, image_shape):
        if sum(isinstance(t, T.HFlipTransform) for t in transforms.transforms) % 2 == 1:
            for obj in dataset_dict.get("annotations", []):
                q, pos = divmod(obj["category_id"], 8)      # ids contíguos 0..31 = q*8+pos
                obj["category_id"] = (q ^ 1) * 8 + pos       # 0<->1, 2<->3
        super()._transform_annotations(dataset_dict, transforms, image_shape)


class Trainer(DefaultTrainer):
    @classmethod
    def build_train_loader(cls, cfg):
        return build_detection_train_loader(cfg, mapper=FlipSwapMapper(cfg, is_train=True))

    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        return COCOEvaluator(dataset_name, output_dir=output_folder or os.path.join(cfg.OUTPUT_DIR, "eval"))


def config_treino():
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
    cfg.DATASETS.TRAIN = ("dentex_train",)
    cfg.DATASETS.TEST = ("dentex_val",)
    cfg.DATALOADER.NUM_WORKERS = 2
    cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")

    # Mesma configuração do treino original (célula 15 do notebook antigo; metrics.json: 20.000 iterações)
    cfg.SOLVER.IMS_PER_BATCH = 2
    cfg.SOLVER.BASE_LR = 0.002
    cfg.SOLVER.MAX_ITER = 20_000
    cfg.SOLVER.STEPS = (14_000, 18_000)
    cfg.SOLVER.GAMMA = 0.1
    cfg.SOLVER.WARMUP_ITERS = 1_000
    cfg.SOLVER.WARMUP_FACTOR = 1.0 / 1_000
    cfg.SOLVER.CHECKPOINT_PERIOD = 2_000
    cfg.INPUT.MIN_SIZE_TRAIN = (640, 672, 704, 736, 768, 800)
    cfg.INPUT.MIN_SIZE_TRAIN_SAMPLING = "choice"
    cfg.INPUT.MIN_SIZE_TEST = 800
    cfg.INPUT.RANDOM_FLIP = "horizontal"               # o flip continua; o mapper troca o rótulo
    cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 512
    cfg.MODEL.ROI_HEADS.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = SCORE_MIN

    if TESTE_RAPIDO:
        cfg.SOLVER.MAX_ITER = 200
        cfg.SOLVER.STEPS = (140, 180)
        cfg.SOLVER.WARMUP_ITERS = 50
        cfg.SOLVER.CHECKPOINT_PERIOD = 100
    cfg.OUTPUT_DIR = PATHS["treino_out"]
    return cfg


def frac_q1q4_esquerda(mapper, n_imgs=40, repeticoes=4):
    """Fração dos dentes Q1/Q4 cujo centro fica na metade esquerda da imagem, depois do mapper."""
    dicts = DatasetCatalog.get("dentex_train")[:n_imgs]
    esq = tot = 0
    for d in dicts:
        for _ in range(repeticoes):
            out = mapper(copy.deepcopy(d))
            inst = out["instances"]
            W = out["image"].shape[2]
            q = inst.gt_classes.numpy() // 8
            cx = inst.gt_boxes.get_centers()[:, 0].numpy()
            sel = np.isin(q, [0, 3])
            esq += int((cx[sel] < W / 2).sum())
            tot += int(sel.sum())
    return esq / max(tot, 1)


if RODAR_FASE2:
    cfg_teste = config_treino()
    f_padrao = frac_q1q4_esquerda(DatasetMapper(cfg_teste, is_train=True))
    f_swap = frac_q1q4_esquerda(FlipSwapMapper(cfg_teste, is_train=True))
    print(f"Q1/Q4 na esquerda da imagem:  DatasetMapper padrão = {f_padrao:.1%}  |  FlipSwapMapper = {f_swap:.1%}")
    assert f_swap >= 0.90, "FlipSwapMapper NÃO está trocando o rótulo direito — não treine!"
    print("OK: o FlipSwapMapper mantém o lado consistente com o número.")

### 2.4 Treino (~4 h numa T4 na rodada completa)

Mesma configuração do treino original. Só muda o mapper. Retoma do último checkpoint se a pasta de saída já tiver um, salvo a cada 2.000 iterações.
No fim, o Detectron2 roda o `COCOEvaluator` no val (mAP bbox e segm, em `output_maskrcnn_flipswap/eval/`).

In [ ]:
if RODAR_FASE2:
    cfg = config_treino()
    os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
    print(f"Treinando {cfg.SOLVER.MAX_ITER} iterações → {cfg.OUTPUT_DIR}")
    t0 = time.time()
    trainer = Trainer(cfg)
    trainer.resume_or_load(resume=True)
    trainer.train()
    PESOS_NOVOS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
    print(f"\nTreino concluído em {(time.time() - t0) / 3600:.2f} h. Pesos: {PESOS_NOVOS}")
    del trainer
    torch.cuda.empty_cache()

### 2.5 Avaliação do modelo novo e comparação

In [ ]:
def ic95_ganho(antes, depois, n_boot=2000):
    """IC 95% (bootstrap por imagem) da diferença de F1, depois − antes."""
    rng = np.random.default_rng(0)
    n = len(antes)
    d = []
    for _ in range(n_boot):
        s = rng.integers(0, n, n)
        d.append(f1([depois[i] for i in s])[2] - f1([antes[i] for i in s])[2])
    return float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))


if RODAR_FASE2:
    t0 = time.time()
    RES_NOVO, POR_IMG_NOVO = avaliar_modelo(PESOS_NOVOS, VAL_IDS)
    imprimir(f"MODELO ATUAL ({len(VAL_IDS)} imagens)", RES_ATUAL)
    imprimir(f"MODELO NOVO — flip com troca de rótulo ({(time.time() - t0) / 60:.1f} min)", RES_NOVO)

    print("\nGanho de F1 (novo − atual) e IC 95% por bootstrap:")
    ganhos = {}
    for k, nome in [("A", "A_cru_resolucao_treino"), ("B", "B_novo_512"), ("C", "C_novo_512_sem_linha_media")]:
        lo, hi = ic95_ganho(POR_IMG_ATUAL[k], POR_IMG_NOVO[k])
        g = RES_NOVO[nome]["F1"] - RES_ATUAL[nome]["F1"]
        ganhos[nome] = dict(ganho=g, ic95=[lo, hi])
        print(f"  {nome:<28} {g:+6.1%}  (IC 95%: {lo:+.1%} a {hi:+.1%})")

    MELHOROU = RES_NOVO["B_novo_512"]["F1"] > RES_ATUAL["B_novo_512"]["F1"]
    efeito_linha_media = RES_NOVO["B_novo_512"]["F1"] - RES_NOVO["C_novo_512_sem_linha_media"]["F1"]
    USAR_LINHA_MEDIA = efeito_linha_media >= 0
    print(f"\nModelo novo melhor no pré-rótulo (B)? {'SIM' if MELHOROU else 'NÃO'}")
    print(f"Efeito da linha média no modelo novo (B − C): {efeito_linha_media:+.1%} → "
          f"{'manter' if USAR_LINHA_MEDIA else 'desligar'} a correção de lado")

    json.dump({
        "modo": "teste_rapido" if TESTE_RAPIDO else "completo",
        "n_imagens_val": len(VAL_IDS),
        "pesos_atual": PATHS["pesos_atual"], "pesos_novo": PESOS_NOVOS,
        "atual": RES_ATUAL, "novo": RES_NOVO, "ganho_f1": ganhos,
        "melhorou": MELHOROU, "efeito_linha_media_modelo_novo": efeito_linha_media,
    }, open(PATHS["fase2_resultados"], "w"), indent=2)
    print(f"\nResultados salvos em {PATHS['fase2_resultados']}")

### 2.6 Regerar os pré-rótulos com o modelo novo (só se ele for melhor)

In [ ]:
if RODAR_FASE2 and REGERAR_SE_MELHORAR:
    if MELHOROU:
        print(f"Regerando com o modelo novo (linha média: {'sim' if USAR_LINHA_MEDIA else 'não'})...")
        prelabel_novo(PATHS["ds_images"], PATHS["pseudo_novo_v2"], PESOS_NOVOS,
                      por_pasta=3 if TESTE_RAPIDO else None, usar_linha_media=USAR_LINHA_MEDIA)
        visualizar_jsons(PATHS["pseudo_novo_v2"], num_samples=4)
        zipar(PATHS["pseudo_novo_v2"])
    else:
        print("O modelo novo não melhorou o pré-rótulo no DENTEX val: fica valendo o pseudo_labels_novo.zip da fase 1.")

In [ ]:
print("Arquivos para baixar (painel Output):")
for p in ["/kaggle/working/pseudo_labels_novo.zip", "/kaggle/working/pseudo_labels_novo_v2.zip",
          PATHS["fase2_resultados"], os.path.join(PATHS["treino_out"], "model_final.pth"),
          os.path.join(PATHS["treino_out"], "metrics.json")]:
    if os.path.exists(p):
        print(f"  {os.path.getsize(p) / 1e6:8.1f} MB  {p}")